# Agent Evaluation

This notebook adds occupied thermal comfort and cross-building fairness to the carbon-aware MARLISA reward, then compares all five approaches developed so far:

1. `BasicRBC`;
2. independent SAC;
3. MARLISA;
4. carbon-aware MARLISA; and
5. fair carbon-aware MARLISA.

The main question is whether the fair controller can retain useful carbon reduction while substantially improving the experience of the worst-served building.


In [23]:
from __future__ import annotations
import sys
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUTPUTS_ROOT = PROJECT_ROOT / 'outputs'


CONTROLLERS = {
    "rbc": {"label": "RBC", "color": "#64748B"},
    "independent_sac": {"label": "Independent SAC", "color": "#2563EB"},
    "marlisa": {"label": "MARLISA", "color": "#0F9D8A"},
    "carbon_aware_marlisa": { "label": "Carbon-aware MARLISA", "color": "#F59E0B"},
    "fair_carbon_aware_marlisa": {"label": "Fair carbon-aware MARLISA","color": "#7C3AED"},
}
model_list = ["rbc", "independent_sac", "marlisa", "carbon_aware_marlisa", "fair_carbon_aware_marlisa"]
COMFORT_BAND_CELSIUS = 2.0

# Load Data

In [ ]:
@dataclass
class RunData:
    """Files loaded for one saved controller run."""

    key: str
    label: str
    color: str
    kpis: pd.DataFrame
    observations: pd.DataFrame
    actions: pd.DataFrame
    metadata: dict[str, Any]

def load_runs(outputs_root: Path = OUTPUTS_ROOT) -> dict[str, RunData]:
    """Load canonical runs, skipping controllers without KPI files."""
    runs: dict[str, RunData] = {}

    for key, display in CONTROLLERS.items():
        folder = outputs_root / key
        kpi_path = folder / "kpis.csv"
        if not kpi_path.is_file():
            continue

        metadata_path = folder / "metadata.json"
        metadata = (
            json.loads(metadata_path.read_text(encoding="utf-8"))
            if metadata_path.is_file()
            else {}
        )
        runs[key] = RunData(
            key=key,
            label=display["label"],
            color=display["color"],
            kpis=pd.read_csv(kpi_path),
            observations= pd.read_csv(folder / "observations.csv"),
            actions= pd.read_csv(folder / "actions.csv"),
            metadata=metadata,
        )

    if not runs:
        raise FileNotFoundError(
            f"No canonical controller KPI files were found under {outputs_root}."
        )

    return runs
##-------------------------##

runs = load_runs()


# Helper Functions

In [40]:
def build_summary(runs: dict[str, RunData]) -> pd.DataFrame:
    """Calculate one carbon and comfort summary row for every run."""
    rows: list[dict[str, Any]] = []

    for key, run in runs.items():
        district_carbon = run.kpis.loc[
            (run.kpis["cost_function"] == "carbon_emissions_total")
            & (run.kpis["level"] == "district"),
            "value",
        ]
        discomfort = run.kpis.loc[
            (run.kpis["cost_function"] == "discomfort_proportion")
            & (run.kpis["level"] == "building"),
            ["name", "value"],
        ].dropna()
        if len(district_carbon) != 1 or discomfort.empty:
            continue

        worst_value = float(discomfort["value"].max())
        best_value = float(discomfort["value"].min())
        worst_names = discomfort.loc[
            np.isclose(
                discomfort["value"], worst_value, rtol=0.0, atol=1e-12
            ),
            "name",
        ].astype(str)
        rows.append(
            {
                "model": key,
                "controller": run.label,
                "color": run.color,
                "carbon_ratio": float(district_carbon.iloc[0]),
                "average_discomfort": float(discomfort["value"].mean()),
                "best_discomfort": best_value,
                "worst_discomfort": worst_value,
                "discomfort_gap": worst_value - best_value,
                "worst_building": ", ".join(worst_names),
                "seed": run.metadata.get("random_seed"),
                "episodes": run.metadata.get("episodes"),
            }
        )

    return pd.DataFrame(rows)
##-------------------------##

def building_discomfort(runs: dict[str, RunData]) -> pd.DataFrame:
    """Return building-level discomfort rows for all runs."""
    frames: list[pd.DataFrame] = []
    for key, run in runs.items():
        values = run.kpis.loc[
            (run.kpis["cost_function"] == "discomfort_proportion")
            & (run.kpis["level"] == "building"),
            ["name", "value"],
        ].copy()
        values["model"] = key
        values["controller"] = run.label
        frames.append(values)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

##-------------------------##

summary = build_summary(runs)

In [39]:
def _base_layout(fig: go.Figure, *, height: int = 390) -> go.Figure:
    fig.update_layout(
        height=height,
        margin=dict(l=54, r=24, t=58, b=50),
        paper_bgcolor="#FFFFFF",
        plot_bgcolor="#F8FAFC",
        font=dict(family="Inter, Arial, sans-serif", color="#172033"),
        hoverlabel=dict(bgcolor="#172033", font_color="#FFFFFF"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    )
    fig.update_xaxes(gridcolor="#E2E8F0", zeroline=False)
    fig.update_yaxes(gridcolor="#E2E8F0", zeroline=False)
    return fig


## a) Summary Statistics


In [61]:
summary = build_summary(runs)
req=['model',  'carbon_ratio', 'average_discomfort',
       'best_discomfort', 'worst_discomfort', 'discomfort_gap',
       'worst_building', 'episodes']
display(summary[req])

,model,carbon_ratio,average_discomfort,best_discomfort,worst_discomfort,discomfort_gap,worst_building,episodes
0,rbc,1.889587,0.946599,0.908411,0.974719,0.066308,Building_1,NaN
1,independent_sac,0.445374,0.979771,0.977570,0.981742,0.004171,Building_1,3.0
2,marlisa,0.684745,0.510537,0.306667,0.845506,0.538839,Building_1,3.0
3,carbon_aware_marlisa,0.449130,0.979148,0.975701,0.981742,0.006041,Building_1,3.0
4,fair_carbon_aware_marlisa,0.872638,0.201070,0.133333,0.336449,0.203115,Building_2,3.0


## b) Carbon Performance against the reference

In [46]:
def carbon_bar_figure(summary: pd.DataFrame, focus: str) -> go.Figure:
    """Carbon ratio bars with the no-control reference marked at 1.0."""
    frame = summary.sort_values("carbon_ratio", ascending=True)
    colors = [
        CONTROLLERS[m]["color"] if m == focus else "#CBD5E1"
        for m in frame["model"]
    ]
    fig = go.Figure(
        go.Bar(
            x=frame["carbon_ratio"],
            y=frame["controller"],
            orientation="h",
            marker_color=colors,
            text=frame["carbon_ratio"].map(lambda x: f"{x:.3f}"),
            textposition="outside",
            customdata=(1.0 - frame["carbon_ratio"]) * 100.0,
            hovertemplate=(
                "%{y}<br>Carbon ratio: %{x:.3f}<br>"
                "Change vs no-control: %{customdata:+.1f}%<extra></extra>"
            ),
        )
    )
    fig.add_vline(
        x=1.0,
        line_dash="dash",
        line_color="#DC2626",
        annotation_text="No-control reference",
        annotation_position="top right",
    )
    fig.update_layout(title="Carbon performance against the reference", showlegend=False)
    fig.update_xaxes(title="Carbon-emissions ratio · lower is better", rangemode="tozero")
    fig.update_yaxes(title=None)
    return _base_layout(fig)

carbon_bar_figure_output = carbon_bar_figure(summary, focus="fair_carbon_aware_marlisa")
carbon_bar_figure_output

## c) The Central Trade off: Carbon Emissions vs Discomfort

In [55]:
def tradeoff_figure(summary: pd.DataFrame, focus: str) -> go.Figure:
    """Carbon versus worst-building discomfort trade-off."""
    fig = go.Figure()
    for row in summary.itertuples(index=False):
        selected = row.model == focus
        fig.add_trace(
            go.Scatter(
                x=[row.carbon_ratio],
                y=[row.worst_discomfort],
                mode="markers+text",
                text=[row.controller],
                textposition="top center",
                marker=dict(
                    size=18 if selected else 13,
                    color=row.color,
                    line=dict(color="#FFFFFF", width=3 if selected else 1.5),
                    opacity=1.0 if selected else 0.78,
                ),
                customdata=[[row.average_discomfort, row.discomfort_gap]],
                hovertemplate=(
                    "%{text}<br>Carbon ratio: %{x:.3f}<br>"
                    "Worst discomfort: %{y:.1%}<br>"
                    "Average discomfort: %{customdata[0]:.1%}<br>"
                    "Gap: %{customdata[1]:.1%}<extra></extra>"
                ),
                showlegend=False,
            )
        )
    fig.add_hline(y=0.5, line_dash="dash", line_color="#DC2626")
    fig.add_vline(x=1.0, line_dash="dash", line_color="#DC2626")
    fig.add_annotation(
        x=0.02,
        y=0.04,
        xref="paper",
        yref="paper",
        text="Preferred Quadrant (direction ↙)",
        showarrow=False,
        font=dict(color="#166534", size=12),
        bgcolor="#DCFCE7",
        borderpad=5,
    )
    fig.update_layout(title="The central trade-off: carbon versus protected comfort")
    fig.update_xaxes(title="Carbon-emissions ratio", rangemode="tozero")
    fig.update_yaxes(
        title="Worst-building discomfort",
        range=[0, 1.2],
        tickformat=".0%",
    )
    return _base_layout(fig)

tradeoff_figure_output = tradeoff_figure(summary, focus="fair_carbon_aware_marlisa")
tradeoff_figure_output

## d) Which Building experience discomfort?

In [56]:
def discomfort_by_building_figure(
    runs: dict[str, RunData], focus: str
) -> go.Figure:
    """Grouped building-level discomfort chart."""
    values = building_discomfort(runs)
    fig = go.Figure()
    for key, run in runs.items():
        part = values.loc[values["model"] == key]
        fig.add_trace(
            go.Bar(
                x=part["name"],
                y=part["value"],
                name=run.label,
                marker_color=run.color,
                opacity=1.0 if key == focus else 0.55,
                hovertemplate="%{x}<br>Discomfort: %{y:.1%}<extra>"
                + run.label
                + "</extra>",
            )
        )
    fig.update_layout(
        title="Who experiences discomfort?",
        barmode="group",
    )
    fig.update_xaxes(title=None)
    fig.update_yaxes(
        title="Occupied time outside comfort band",
        range=[0, 1.03],
        tickformat=".0%",
    )
    return _base_layout(fig, height=420)

discomfort_by_building_figure_output = discomfort_by_building_figure(runs, focus="fair_carbon_aware_marlisa")
discomfort_by_building_figure_output

In [57]:
def comfort_range_figure(summary: pd.DataFrame, focus: str) -> go.Figure:
    """Best-to-worst comfort ranges with average markers."""
    frame = summary.sort_values("average_discomfort", ascending=False)
    fig = go.Figure()
    for row in frame.itertuples(index=False):
        opacity = 1.0 if row.model == focus else 0.48
        fig.add_trace(
            go.Scatter(
                x=[row.best_discomfort, row.worst_discomfort],
                y=[row.controller, row.controller],
                mode="lines+markers",
                line=dict(color=row.color, width=7),
                marker=dict(color=row.color, size=11),
                opacity=opacity,
                hovertemplate="%{y}<br>%{x:.1%}<extra></extra>",
                showlegend=False,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=[row.average_discomfort],
                y=[row.controller],
                mode="markers",
                marker=dict(
                    color="#FFFFFF",
                    line=dict(color=row.color, width=3),
                    size=13,
                    symbol="diamond",
                ),
                opacity=opacity,
                hovertemplate=(
                    f"{row.controller}<br>Average: "
                    "%{x:.1%}<extra></extra>"
                ),
                showlegend=False,
            )
        )
    fig.add_annotation(
        x=0.99,
        y=1.12,
        xref="paper",
        yref="paper",
        text="Ends = best/worst · diamond = average",
        showarrow=False,
        font=dict(size=11, color="#64748B"),
    )
    fig.update_layout(title="Comfort level and inequality must be read together")
    fig.update_xaxes(
        title="Building discomfort proportion",
        range=[0, 1.03],
        tickformat=".0%",
    )
    fig.update_yaxes(title=None)
    return _base_layout(fig, height=420)

comfort_range_figure_output = comfort_range_figure(summary, focus="fair_carbon_aware_marlisa")
comfort_range_figure_output